This notebook creates a heatmap of pairwise cluster correlations between Dumitru et al. and Liu et al. datasets.

Input: .h5ad with annotated cell types in ../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_nsc_annotations.h5ad  
Output: figure

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
sc.settings.set_figure_params(dpi=300)
import matplotlib.pyplot as plt
plt.rcParams['axes.grid'] = False

In [ ]:
dumitru = sc.read_h5ad('../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_nsc_annotations.h5ad')
dumitru

In [ ]:
dumitru.obs['cell_type'].value_counts()

In [ ]:
liu_raw = sc.read_h5ad('../Data/Liu/hNSPC_raw_counts.h5ad')
liu_raw

In [ ]:
liu_processed = sc.read_h5ad('../Data/Liu/hNSPC.h5ad')
liu_processed

In [ ]:
liu_processed.obs['annotation'].value_counts()

In [ ]:
# Get cell barcodes/indices from liu_processed
liu_processed_cells = liu_processed.obs.index

# Filter liu_raw to only keep cells that are in liu_processed
liu_raw_filtered = liu_raw[liu_raw.obs.index.isin(liu_processed_cells)].copy()

# Reorder liu_raw_filtered to match the order of liu_processed
liu_raw_filtered = liu_raw_filtered[liu_processed_cells]

# Transfer annotation from liu_processed to liu_raw_filtered
liu_raw_filtered.obs['annotation'] = liu_processed.obs['annotation']

liu = liu_raw_filtered
liu

In [ ]:
# Clean up variable names
liu.var_names = liu.var_names.str.split('_').str[1]
mask = liu.var_names.notna()
liu = liu[:, mask].copy()
liu.var_names_make_unique()
liu.var_names

In [ ]:
def correlate_datasets(adata1, adata2, group1, group2,
                      layer1='X', layer2='X',
                      order1=None, order2=None,
                      title=None, xlabel=None, ylabel=None,
                      figsize=(7, 5),
                      show_values=True):
    import scanpy as sc
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
    from scipy.stats import pearsonr, zscore
    import numpy as np
    plt.rcParams['axes.grid'] = False
    
    # Copy and normalize data
    def normalize_adata(adata, layer):
        adata_norm = adata.copy()
        if layer != 'X':
            adata_norm.X = adata_norm.layers[layer].copy()
        sc.pp.normalize_total(adata_norm, target_sum=1e4)
        sc.pp.log1p(adata_norm)
        return adata_norm
    
    adata1_norm = normalize_adata(adata1, layer1)
    adata2_norm = normalize_adata(adata2, layer2)
    
    # Find common genes and subset
    common_genes = adata1_norm.var_names.intersection(adata2_norm.var_names)
    print(f"Common genes: {len(common_genes)}")
    
    adata1_sub = adata1_norm[:, common_genes]
    adata2_sub = adata2_norm[:, common_genes]
    
    # Calculate mean expression per cell type
    def get_means(adata, group):
        df = adata.to_df()
        df['cluster'] = adata.obs[group]
        return df.groupby('cluster').mean()
    
    means1 = get_means(adata1_sub, group1)
    means2 = get_means(adata2_sub, group2)
    
    # Z-score normalize genes with NaN handling
    def safe_zscore(df, axis=0):
        result = df.apply(zscore, axis=axis)
        # Report genes with zero variance
        nan_genes = result.columns[result.isna().any(axis=0)]
        if len(nan_genes) > 0:
            print(f"Genes with zero variance (will be excluded): {len(nan_genes)}")
        return result.dropna(axis=1)  # Remove genes with NaN values
    
    means1_zscore = safe_zscore(means1, axis=0)
    means2_zscore = safe_zscore(means2, axis=0)
    
    # Keep only genes present in both datasets after NaN removal
    common_genes_final = means1_zscore.columns.intersection(means2_zscore.columns)
    means1_zscore = means1_zscore[common_genes_final]
    means2_zscore = means2_zscore[common_genes_final]
    print(f"Genes after removing zero-variance genes: {len(common_genes_final)}")
    
    # Calculate correlations between cell types using z-scored data
    corr_matrix = []
    for type1 in means1_zscore.index:
        row = []
        for type2 in means2_zscore.index:
            corr, _ = pearsonr(means1_zscore.loc[type1], means2_zscore.loc[type2])
            row.append(corr)
        corr_matrix.append(row)
    
    # Create correlation dataframe
    corr_df = pd.DataFrame(corr_matrix, index=means1.index, columns=means2.index)
    
    # Reorder if specified
    if order1:
        available_types1 = [ct for ct in order1 if ct in corr_df.index]
        corr_df = corr_df.reindex(available_types1)
    if order2:
        available_types2 = [ct for ct in order2 if ct in corr_df.columns]
        corr_df = corr_df.reindex(columns=available_types2)

    # Plot heatmap
    plt.figure(figsize=figsize)
    sns.heatmap(
        corr_df,
        annot=show_values,
        cmap='RdBu_r',
        center=0,
        fmt='.2f' if show_values else '',
        square=True,
        linewidths=0.5,
        cbar_kws={'label': 'Pearson correlation'}
    )
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()
    
    return corr_df

In [ ]:
corr_matrix = correlate_datasets(
    adata1=dumitru, adata2=liu, 
    group1='cell_type', group2='annotation',
    layer1='counts', layer2='X',
    title="Dumitru vs Liu",
    xlabel="Liu et al. cell types",
    ylabel="Dumitru et al. cell types",
    figsize=(12, 8)
)

In [ ]:
dumitru_order = ['nsc_01', 'nsc_02', 'nsc_03', 'nsc_04', 'nsc_05']
liu_order = ['Astrocyte', 'Ventricular radial glia', 'Outer radial glia', 'Pre-OPC', 'Radial glia (G2M phase)', 'Radial glia (S phase)']

corr_matrix = correlate_datasets(
    adata1=dumitru, adata2=liu, 
    group1='cell_type', group2='annotation',
    layer1='counts', layer2='X',
    order1=dumitru_order, order2=liu_order,
    xlabel="Liu et al. cell types",
    ylabel="Dumitru et al. cell types",
    figsize=(6, 6)
)

In [ ]:
dumitru_order = ['nsc_01', 'nsc_02', 'nsc_03', 'nsc_04', 'nsc_05']
liu_order = ['Astrocyte', 'Ventricular radial glia', 'Outer radial glia', 'Pre-OPC', 'Radial glia (G2M phase)', 'Radial glia (S phase)']

corr_matrix = correlate_datasets(
    adata1=dumitru, adata2=liu, 
    group1='cell_type', group2='annotation',
    layer1='counts', layer2='X',
    order1=dumitru_order, order2=liu_order,
    xlabel="Liu et al. cell types",
    ylabel="Dumitru et al. cell types",
    show_values=False,
    figsize=(6, 6)
)